# Pipeline Logger — Centralized Audit & Telemetry
### Structured logging for the Cross-Industry Accelerator pipeline

This utility provides **pipeline-level logging** that extends `ZT_Security_Utils` audit logging.
Import via: `%run Pipeline_Logger`

| Capability | Description |
|---|---|
| **Step tracking** | Start/complete/fail each notebook with timing |
| **Record counts** | Track rows ingested, tables created, artifacts deployed |
| **Error capture** | Structured error logging with context and stack traces |
| **Artifact registry** | Log every Fabric artifact created with name, type, and ID |
| **Pipeline summary** | Full end-to-end report for audit and compliance |
| **Lakehouse persistence** | Write the full log to Delta tables for long-term audit |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PIPELINE LOGGER — Core logging engine
# ════════════════════════════════════════════════════════════════════════

import datetime
import json
import traceback
import uuid

# ── Pipeline run identity ──────────────────────────────────────────────
PIPELINE_RUN_ID = str(uuid.uuid4())[:12]
PIPELINE_START_TIME = datetime.datetime.utcnow()
PIPELINE_STATUS = "RUNNING"

# ── In-memory log stores ──────────────────────────────────────────────
_PIPELINE_LOG = []          # All pipeline events (superset)
_STEP_LOG = {}              # step_name → {start, end, status, details, records, errors}
_ARTIFACT_REGISTRY = []     # Every Fabric artifact created
_RECORD_COUNTS = {}         # step → {table: row_count}
_ERROR_LOG = []             # Structured errors with context

# ── Notebook step definitions ─────────────────────────────────────────
PIPELINE_STEPS = [
    {"step": "00_Industry_Config",       "order": 0, "phase": "Configure",    "description": "Set industry, auto-discover tables, create Lakehouse"},
    {"step": "01_Data_Ingestion",        "order": 1, "phase": "Ingest",       "description": "Profile schemas, detect quality issues, validate data"},
    {"step": "02_Load_Lakehouse",        "order": 2, "phase": "Integrate",    "description": "Load dim/fact → Lakehouse Delta; events → Eventhouse"},
    {"step": "03_Load_Warehouse",        "order": 3, "phase": "Integrate",    "description": "Load all tables → Warehouse SQL (auto DDL)"},
    {"step": "04_Create_Semantic_Model", "order": 4, "phase": "Model",        "description": "Create semantic model with measures & relationships"},
    {"step": "05_Create_Ontology",       "order": 5, "phase": "Intelligence", "description": "Create Fabric IQ ontology with data bindings"},
    {"step": "06_Create_Data_Agent",     "order": 6, "phase": "Intelligence", "description": "Deploy QA Agent + Ops Agent with industry prompts"},
    {"step": "07_Create_Dashboards",     "order": 7, "phase": "Visualize",    "description": "Create KQL real-time dashboard + Power BI report"},
]


def _now():
    return datetime.datetime.utcnow().isoformat() + "Z"


def _elapsed(start_time):
    """Return human-readable elapsed time."""
    delta = datetime.datetime.utcnow() - start_time
    total_secs = int(delta.total_seconds())
    mins, secs = divmod(total_secs, 60)
    return f"{mins}m {secs}s" if mins else f"{secs}s"


# ═══════════════════════════════════════════════════════════════════════
# STEP LIFECYCLE — start / complete / fail
# ═══════════════════════════════════════════════════════════════════════

def pipeline_step_start(step_name, description=""):
    """Mark a pipeline step as started."""
    _STEP_LOG[step_name] = {
        "step": step_name,
        "start": datetime.datetime.utcnow(),
        "end": None,
        "status": "RUNNING",
        "description": description,
        "records_loaded": {},
        "artifacts_created": [],
        "errors": [],
    }
    _pipeline_event("STEP_START", step_name, description)
    print(f"\n{'━'*65}")
    print(f"▶ STEP: {step_name}")
    if description:
        print(f"  {description}")
    print(f"  Started: {_now()}")
    print(f"{'━'*65}")


def pipeline_step_complete(step_name, summary=""):
    """Mark a pipeline step as successfully completed."""
    entry = _STEP_LOG.get(step_name)
    if not entry:
        _pipeline_event("STEP_COMPLETE", step_name, "Warning: step_start not called")
        return
    entry["end"] = datetime.datetime.utcnow()
    entry["status"] = "SUCCESS"
    elapsed = _elapsed(entry["start"])
    records = entry["records_loaded"]
    total_rows = sum(records.values()) if records else 0
    artifacts = len(entry["artifacts_created"])

    detail_parts = [f"elapsed={elapsed}"]
    if total_rows:
        detail_parts.append(f"rows={total_rows:,}")
    if artifacts:
        detail_parts.append(f"artifacts={artifacts}")
    if summary:
        detail_parts.append(summary)

    _pipeline_event("STEP_COMPLETE", step_name, " | ".join(detail_parts))
    print(f"\n  ✅ {step_name} completed in {elapsed}")
    if total_rows:
        print(f"     Records: {total_rows:,} rows across {len(records)} tables")
    if artifacts:
        print(f"     Artifacts: {artifacts} created")


def pipeline_step_fail(step_name, error, fatal=True):
    """Mark a pipeline step as failed with error context."""
    entry = _STEP_LOG.get(step_name, {})
    if entry:
        entry["end"] = datetime.datetime.utcnow()
        entry["status"] = "FAILED"
        if isinstance(entry.get("errors"), list):
            entry["errors"].append(str(error))

    error_entry = {
        "timestamp": _now(),
        "run_id": PIPELINE_RUN_ID,
        "step": step_name,
        "error": str(error),
        "traceback": traceback.format_exc(),
        "fatal": fatal,
    }
    _ERROR_LOG.append(error_entry)
    _pipeline_event("STEP_FAILED", step_name, str(error), status="ERROR")
    print(f"\n  ❌ {step_name} FAILED: {error}")
    if fatal:
        print(f"     Pipeline will stop. Check error log for details.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DATA RECORD TRACKING
# ═══════════════════════════════════════════════════════════════════════

def log_records_loaded(step_name, table_name, row_count, target="", status="OK"):
    """Log row counts for a table load operation."""
    entry = _STEP_LOG.get(step_name)
    if entry:
        entry["records_loaded"][table_name] = row_count

    if step_name not in _RECORD_COUNTS:
        _RECORD_COUNTS[step_name] = {}
    _RECORD_COUNTS[step_name][table_name] = row_count

    detail = f"{row_count:,} rows"
    if target:
        detail += f" → {target}"
    _pipeline_event("RECORDS_LOADED", table_name, detail, status=status)


def log_records_error(step_name, table_name, error, target=""):
    """Log a failed table load."""
    entry = _STEP_LOG.get(step_name)
    if entry:
        entry["errors"].append(f"{table_name}: {error}")

    detail = f"FAILED: {error}"
    if target:
        detail += f" (target: {target})"
    _pipeline_event("RECORDS_ERROR", table_name, detail, status="ERROR")
    _ERROR_LOG.append({
        "timestamp": _now(), "run_id": PIPELINE_RUN_ID, "step": step_name,
        "error": f"{table_name}: {error}", "traceback": "", "fatal": False,
    })


# ═══════════════════════════════════════════════════════════════════════
# ARTIFACT REGISTRY
# ═══════════════════════════════════════════════════════════════════════

def log_artifact_created(step_name, artifact_type, artifact_name, artifact_id="", details=""):
    """Register a Fabric artifact created during the pipeline."""
    artifact = {
        "timestamp": _now(),
        "run_id": PIPELINE_RUN_ID,
        "step": step_name,
        "type": artifact_type,
        "name": artifact_name,
        "id": artifact_id,
        "details": details,
    }
    _ARTIFACT_REGISTRY.append(artifact)

    entry = _STEP_LOG.get(step_name)
    if entry:
        entry["artifacts_created"].append(f"{artifact_type}: {artifact_name}")

    _pipeline_event("ARTIFACT_CREATED", artifact_name, f"type={artifact_type}" + (f" id={artifact_id}" if artifact_id else ""))
    print(f"  📦 Created: {artifact_type} → {artifact_name}" + (f" ({artifact_id[:8]}...)" if artifact_id else ""))


def log_artifact_deleted(step_name, artifact_type, artifact_name, artifact_id=""):
    """Log deletion of an existing artifact (before recreation)."""
    _pipeline_event("ARTIFACT_DELETED", artifact_name, f"type={artifact_type}" + (f" id={artifact_id}" if artifact_id else ""), status="INFO")


# ═══════════════════════════════════════════════════════════════════════
# CORE EVENT LOGGER
# ═══════════════════════════════════════════════════════════════════════

def _pipeline_event(action, target, details="", status="OK"):
    """Append a structured event to the pipeline log."""
    event = {
        "timestamp": _now(),
        "run_id": PIPELINE_RUN_ID,
        "action": action,
        "target": target,
        "details": details,
        "status": status,
    }
    _PIPELINE_LOG.append(event)


def log_pipeline_event(action, target, details="", status="OK"):
    """Public interface — log a custom pipeline event."""
    _pipeline_event(action, target, details, status)
    print(f"  📋 [{status}] {action} → {target}" + (f" ({details})" if details else ""))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PIPELINE SUMMARY & REPORTING
# ═══════════════════════════════════════════════════════════════════════

def get_pipeline_summary():
    """Return structured pipeline execution summary."""
    global PIPELINE_STATUS
    failed_steps = [s for s in _STEP_LOG.values() if s["status"] == "FAILED"]
    PIPELINE_STATUS = "FAILED" if failed_steps else "SUCCESS"

    total_rows = sum(
        sum(step.get("records_loaded", {}).values())
        for step in _STEP_LOG.values()
    )
    total_artifacts = len(_ARTIFACT_REGISTRY)
    total_errors = len(_ERROR_LOG)

    return {
        "run_id": PIPELINE_RUN_ID,
        "status": PIPELINE_STATUS,
        "start_time": PIPELINE_START_TIME.isoformat() + "Z",
        "end_time": _now(),
        "elapsed": _elapsed(PIPELINE_START_TIME),
        "steps_completed": sum(1 for s in _STEP_LOG.values() if s["status"] == "SUCCESS"),
        "steps_failed": len(failed_steps),
        "total_rows_processed": total_rows,
        "total_artifacts_created": total_artifacts,
        "total_errors": total_errors,
        "steps": {
            name: {
                "status": info["status"],
                "elapsed": _elapsed(info["start"]) if info.get("end") else "N/A",
                "rows": sum(info.get("records_loaded", {}).values()),
                "artifacts": len(info.get("artifacts_created", [])),
                "errors": len(info.get("errors", [])),
            }
            for name, info in _STEP_LOG.items()
        },
        "artifacts": _ARTIFACT_REGISTRY,
        "errors": _ERROR_LOG,
    }


def print_pipeline_summary():
    """Print a formatted pipeline execution report."""
    summary = get_pipeline_summary()

    print(f"\n{'═'*70}")
    print(f"PIPELINE EXECUTION REPORT")
    print(f"{'═'*70}")
    print(f"  Run ID:      {summary['run_id']}")
    print(f"  Status:      {summary['status']}")
    print(f"  Started:     {summary['start_time']}")
    print(f"  Finished:    {summary['end_time']}")
    print(f"  Elapsed:     {summary['elapsed']}")
    print(f"")

    # Step summary table
    print(f"  {'Step':<30} {'Status':<10} {'Time':<10} {'Rows':>10} {'Artifacts':>10} {'Errors':>7}")
    print(f"  {'─'*30} {'─'*10} {'─'*10} {'─'*10} {'─'*10} {'─'*7}")
    for name, info in summary["steps"].items():
        status_icon = "✅" if info["status"] == "SUCCESS" else "❌" if info["status"] == "FAILED" else "⏳"
        print(f"  {name:<30} {status_icon} {info['status']:<7} {info['elapsed']:<10} {info['rows']:>10,} {info['artifacts']:>10} {info['errors']:>7}")
    print(f"  {'─'*30} {'─'*10} {'─'*10} {'─'*10} {'─'*10} {'─'*7}")
    print(f"  {'TOTALS':<30} {'':10} {'':10} {summary['total_rows_processed']:>10,} {summary['total_artifacts_created']:>10} {summary['total_errors']:>7}")

    # Artifacts created
    if _ARTIFACT_REGISTRY:
        print(f"\n  ARTIFACTS CREATED ({len(_ARTIFACT_REGISTRY)}):")
        for a in _ARTIFACT_REGISTRY:
            print(f"    📦 [{a['type']}] {a['name']}" + (f"  id={a['id'][:12]}..." if a.get('id') else ""))

    # Errors
    if _ERROR_LOG:
        print(f"\n  ERRORS ({len(_ERROR_LOG)}):")
        for e in _ERROR_LOG:
            fatal_tag = " [FATAL]" if e.get("fatal") else ""
            print(f"    ❌ [{e['step']}]{fatal_tag} {e['error'][:120]}")

    print(f"\n{'═'*70}")
    print(f"  Pipeline {summary['status']} — {summary['steps_completed']}/{summary['steps_completed'] + summary['steps_failed']} steps completed")
    print(f"{'═'*70}")

    return summary


def get_full_pipeline_log():
    """Return the raw event log for persistence."""
    return _PIPELINE_LOG.copy()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# LAKEHOUSE PERSISTENCE — Write audit log to Delta tables
# ═══════════════════════════════════════════════════════════════════════

def persist_pipeline_log(spark_session, lakehouse_schema="dbo", lakehouse_abfs_base=""):
    """Write pipeline log, artifacts, and errors to Lakehouse Delta tables for long-term audit.
    
    Args:
        spark_session: Active Spark session
        lakehouse_schema: Schema name (default: 'dbo')
        lakehouse_abfs_base: Optional ABFS base path for the target lakehouse.
            If provided, writes to ABFS paths (works with non-default lakehouses).
            If empty, uses saveAsTable (writes to default lakehouse).
    """
    try:
        from pyspark.sql import Row

        summary = get_pipeline_summary()

        def _save_delta(df, table_name):
            """Save DataFrame to Delta — uses ABFS path if available, else saveAsTable."""
            if lakehouse_abfs_base:
                path = f"{lakehouse_abfs_base}/Tables/{lakehouse_schema}/{table_name}"
                df.write.mode("append").format("delta").option("mergeSchema", "true").save(path)
            else:
                df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(
                    f"{lakehouse_schema}.{table_name}"
                )

        # ── pipeline_runs ──
        run_row = Row(
            run_id=summary["run_id"],
            status=summary["status"],
            start_time=summary["start_time"],
            end_time=summary["end_time"],
            elapsed=summary["elapsed"],
            steps_completed=summary["steps_completed"],
            steps_failed=summary["steps_failed"],
            total_rows=summary["total_rows_processed"],
            total_artifacts=summary["total_artifacts_created"],
            total_errors=summary["total_errors"],
            industry=globals().get("INDUSTRY", "unknown"),
        )
        _save_delta(spark_session.createDataFrame([run_row]), "pipeline_runs")
        target = lakehouse_abfs_base.split("/")[-1][:12] + "..." if lakehouse_abfs_base else "default"
        print(f"  💾 Persisted pipeline run → {lakehouse_schema}.pipeline_runs ({target})")

        # ── pipeline_events ──
        if _PIPELINE_LOG:
            event_rows = [Row(**e) for e in _PIPELINE_LOG]
            _save_delta(spark_session.createDataFrame(event_rows), "pipeline_events")
            print(f"  💾 Persisted {len(_PIPELINE_LOG)} events → {lakehouse_schema}.pipeline_events")

        # ── pipeline_artifacts ──
        if _ARTIFACT_REGISTRY:
            art_rows = [Row(**a) for a in _ARTIFACT_REGISTRY]
            _save_delta(spark_session.createDataFrame(art_rows), "pipeline_artifacts")
            print(f"  💾 Persisted {len(_ARTIFACT_REGISTRY)} artifacts → {lakehouse_schema}.pipeline_artifacts")

        # ── pipeline_errors ──
        if _ERROR_LOG:
            err_rows = [Row(
                timestamp=e["timestamp"], run_id=PIPELINE_RUN_ID, step=e["step"],
                error=e["error"], traceback_info=e.get("traceback", "")[:2000],
                fatal=e.get("fatal", False),
            ) for e in _ERROR_LOG]
            _save_delta(spark_session.createDataFrame(err_rows), "pipeline_errors")
            print(f"  💾 Persisted {len(_ERROR_LOG)} errors → {lakehouse_schema}.pipeline_errors")

        log_pipeline_event("LOG_PERSISTED", "Lakehouse", f"run={PIPELINE_RUN_ID}")

    except Exception as e:
        print(f"  ⚠ Could not persist pipeline log: {e}")
        print(f"    (In-memory log is still available via get_pipeline_summary())")


print(f"✅ Pipeline Logger loaded — Run ID: {PIPELINE_RUN_ID}")
print(f"   Start time: {PIPELINE_START_TIME.isoformat()}Z")
print(f"   Functions: pipeline_step_start/complete/fail, log_records_loaded, log_artifact_created, print_pipeline_summary, persist_pipeline_log")